# 03 - Raw Four-Table Join and Data Quality




## 1. Setup



In [1]:
%pip install psycopg

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [1]:
from rapidfuzz import fuzz, process

In [2]:
from pathlib import Path
from getpass import getpass

import pandas as pd
import psycopg
from psycopg import sql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Sesuaikan bila Anda membuat PostgreSQL role selain "postgres".
POSTGRES_HOST = "localhost"
POSTGRES_PORT = 5432
POSTGRES_USER = "postgres"

# Nama fisik database PostgreSQL: gunakan lowercase.
AML_DATABASE = "aml"

# Masukkan password ketika notebook meminta.
POSTGRES_PASSWORD = getpass("PostgreSQL password: ")

# Lokasi project dan data raw.
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"

assert RAW_DIR.exists(), f"Folder raw tidak ditemukan: {RAW_DIR}"

print(f"Project root : {ROOT}")
print(f"Raw CSV path : {RAW_DIR}")
print(f"Target DB    : {AML_DATABASE}")

Project root : E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated
Raw CSV path : E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated\data\raw
Target DB    : aml


## 2. Create database connection and load raw tables

Buat koneksi database ke postgresql

In [3]:
# CREATE DATABASE tidak boleh dijalankan di dalam transaction biasa,
# sehingga autocommit wajib aktif.
with psycopg.connect(
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    dbname="postgres",
    autocommit=True,
) as admin_connection:
    
    with admin_connection.cursor() as cursor:
        cursor.execute(
            "SELECT 1 FROM pg_database WHERE datname = %s",
            (AML_DATABASE,),
        )
        database_exists = cursor.fetchone() is not None

        if database_exists:
            print(f"Database '{AML_DATABASE}' sudah ada. Tidak dibuat ulang.")
        else:
            cursor.execute(
                sql.SQL("CREATE DATABASE {} ENCODING 'UTF8'").format(
                    sql.Identifier(AML_DATABASE)
                )
            )
            print(f"Database '{AML_DATABASE}' berhasil dibuat.")


Database 'aml' sudah ada. Tidak dibuat ulang.


In [4]:
aml_url = URL.create(
    drivername="postgresql+psycopg",
    username=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    host=POSTGRES_HOST,
    port=POSTGRES_PORT,
    database=AML_DATABASE,
)

engine = create_engine(
    aml_url,
    pool_pre_ping=True,
)

with engine.connect() as connection:
    database_check = connection.execute(
        text("""
            SELECT
                current_database() AS database_name,
                current_user AS database_user,
                version() AS postgresql_version
        """)
    ).mappings().one()

database_check

{'database_name': 'aml', 'database_user': 'postgres', 'postgresql_version': 'PostgreSQL 18.1 on x86_64-windows, compiled by msvc-19.44.35221, 64-bit'}

In [5]:
with engine.begin() as connection:
    connection.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))

print("Schema raw siap.")

Schema raw siap.


In [6]:
RAW_TABLES = {
    "accounts": {
        "file_name": "accounts.csv",
        "parse_dates": ["opening_date"],
    },
    "counterparties": {
        "file_name": "counterparties.csv",
        "parse_dates": [],
    },
    "customers": {
        "file_name": "customers.csv",
        "parse_dates": ["date_of_birth", "onboarding_date"],
    },
    "sanctions_watchlist": {
        "file_name": "sanctions_watchlist.csv",
        "parse_dates": ["listing_date"],
    },
    "transactions": {
        "file_name": "transactions.csv",
        "parse_dates": ["transaction_timestamp"],
    },
}

In [8]:
MAX_BIND_PARAMETERS = 60_000
MAX_ROWS_PER_INSERT = 1_000

load_summary = []

for table_name, spec in RAW_TABLES.items():
    csv_path = RAW_DIR / spec["file_name"]

    dataframe = pd.read_csv(
        csv_path,
        parse_dates=spec["parse_dates"],
    )

    # method='multi' sends rows x columns parameters per INSERT.
    # Keep each batch safely below PostgreSQL's 65,535-parameter limit.
    safe_chunksize = max(
        1,
        min(
            MAX_ROWS_PER_INSERT,
            MAX_BIND_PARAMETERS // len(dataframe.columns),
        ),
    )

    dataframe.to_sql(
        name=table_name,
        con=engine,
        schema="raw",
        if_exists="replace",
        index=False,
        method="multi",
        chunksize=safe_chunksize,
    )

    load_summary.append(
        {
            "table": f"raw.{table_name}",
            "csv_rows": len(dataframe),
            "csv_columns": len(dataframe.columns),
            "insert_batch_rows": safe_chunksize,
        }
    )

load_summary_df = pd.DataFrame(load_summary)
display(load_summary_df)

,table,csv_rows,csv_columns,insert_batch_rows
0,raw.accounts,15000,11,1000
1,raw.counterparties,5000,16,1000
2,raw.customers,10000,26,1000
3,raw.sanctions_watchlist,1200,23,1000
4,raw.transactions,250000,32,1000


## 5. Join SQL Tables




In [7]:
PROJECT_ROOT = Path(
    r"E:\Trading\V-Teki Project\Anti Money Laundering Detection Updated"
)

sql_path2 = PROJECT_ROOT / "sql" / "join_four_tables.sql"
sql_path_2 = sql_path2.read_text(encoding="utf-8").strip().rstrip(";")

joined_tables = pd.read_sql_query(
    sql=text(sql_path_2),
    con=engine,
)
display(joined_tables.head(8))



,transaction_id,transaction_timestamp,transaction_type,channel,transaction_status,debit_credit,amount,currency,amount_idr_equivalent,purpose_code,...,receiver_party_id,receiver_party_name,receiver_party_address,receiver_party_country,receiver_party_risk_level,receiver_customer_id,receiver_account_id,counterparty_id,beneficiary_name,beneficiary_address
0,TXN0000000001,2025-11-03 00:00:18,SWIFT,Internet,Success,Debit,6211901.22,IDR,6211901.22,OTHER,...,CUS0007767,Rizky Chandra,Jl. Prakoso No. 101,ID,Low,CUS0007767,ACC00003390,INTERNAL_ON_US_TRANSFER,Rizky Chandra,Jl. Prakoso No. 101
1,TXN0000000002,2025-11-03 00:02:26,RTGS,Branch,Success,Debit,1347876.32,IDR,1347876.32,INVESTMENT,...,CP0004049,Gita Adinata,101 Synthetic Avenue,ID,Medium,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0004049,Gita Adinata,101 Synthetic Avenue
2,TXN0000000003,2025-11-03 00:03:18,BI-FAST,ATM,Success,Debit,987476.37,IDR,987476.37,BILL,...,CP0003095,Raka Santoso,92 Synthetic Avenue,ID,Low,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0003095,Raka Santoso,92 Synthetic Avenue
3,TXN0000000004,2025-11-03 00:04:15,Cash,Mobile,Success,Debit,149.95,USD,2429199.44,FAMILY,...,CUS0000366,Indra Adinata,Jl. Gunawan No. 92,SG,Low,CUS0000366,ACC00008341,INTERNAL_ON_US_TRANSFER,Indra Adinata,Jl. Gunawan No. 92
4,TXN0000000005,2025-11-03 00:06:48,Transfer,API,Success,Debit,927217.72,IDR,927217.72,TRADE,...,CP0001839,Citra Santoso,593 Synthetic Avenue,ID,Low,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0001839,Citra Santoso,593 Synthetic Avenue
5,TXN0000000006,2025-11-03 00:09:02,Transfer,Branch,Success,Debit,2097778.34,IDR,2097778.34,INVESTMENT,...,CP0001201,Bima Kurniawan,385 Synthetic Avenue,ID,Medium,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,CP0001201,Bima Kurniawan,385 Synthetic Avenue
6,TXN0000000007,2025-11-03 00:11:11,Cash,Mobile,Success,Debit,569967.57,IDR,569967.57,SALARY,...,CUS0003329,Farah Wibowo,Jl. Rahardjo No. 113,ID,Low,CUS0003329,ACC00007330,INTERNAL_ON_US_TRANSFER,Farah Wibowo,Jl. Rahardjo No. 113
7,TXN0000000008,2025-11-03 00:14:25,BI-FAST,ATM,Success,Debit,4899100.46,IDR,4899100.46,SALARY,...,CUS0005801,Anisa Iskandar,Jl. Rahardjo No. 168,ID,Low,CUS0005801,ACC00012038,INTERNAL_ON_US_TRANSFER,Anisa Iskandar,Jl. Rahardjo No. 168


# Proses Join yang saya lakukan:


Sesuai yang tadi di instruksikan fokus saya ke 4 tabel yaitu accounts, counterparties, customers, dan transactions. Karena fokus dari project AML itu adalah kita monitor transaksi maka transactions akan menjadi tabel utama maka dari itu grain/unit analisis saya adalah </br>
<b> Satu baris = satu transaction_id</b>
<br>
Karena tabel transactions berisi 250.000 transaksi. Untuk query dimulai dari <br>
<b> from raw.transactions as t</b> dengan tujuan hasil akhir tetap 250000 baris satu untuk setiap transaksi. Berikut tabel dan perannya:<br>
1. transactions = Kejadian transaksi: waktu, nominal, negara, device, pengirim, penerima dan sebagainya
2. customer = Profil/KYC customer bank: income, pekerjaan, PEP, risk rating.
3. accoutns = Profil rekening pengirim: tipe rekening, saldo, tanggal buka, risk level.
4. counterparties = Profil pihak eksternal yang bukan customer bank

Kalau ibu liat di file sql disitu saya ada melakukan join tabel customers dua kali. Tapi tabel tersebut punya dua peran berbeda. Misalkan:</br>
<b>
left join raw.customers as sender_customer
    on t.sender_customer_id = sender_customer.customer_id</b><br>
Untuk mengambil profil pengirim/sender
Dan satu lagi </br>
<b> left join raw.customers as receiver_customer
    on t.receiver_customer_id = receiver_customer.customer_id</b>
</br>
Itu untuk ambil profil penerima interna. Contoh:
</br>
Pengirim  : CUS0001001</br>
Penerima : CUS0002002
</br>
Terus untuk join dengan accounts saya pakai dua kondisi:
</br>
<b> left join raw.accounts as sender_account
    on t.sender_account_id = sender_account.account_id
    and t.sender_customer_id = sender_account.customer_id</b>
</br>
Tujuannya buat mastiin rekening tersebut benar-benar milik pengirim transaksi.
Tanpa kondisi kedua, secara teori kita bisa mengambil detail rekening yang ID-nya cocok tetapi pemiliknya tidak sesuai. Dengan dua kondisi ini, join menjadi lebih aman.
</br>
Terus bu tadi pagi kan waktu saya kasih liat hasil join kebetulan ada NaN. Itu ternyata setelah saya telusuri lebih dalam itu ada 2 faktor kondisi bisnis/transaksi berbeda: </br>

| Jenis transaksi | Profil customer penerima | Profil counterparty |
|---|---|---|
| Internal | Ada | Tidak ada |
| Eksternal | Tidak ada | Ada |

</br>
Contoh terjadi transfer internal:
receiver_customer_full_name = "Anisa Santoso"
counterparty_name = NaN
</br>

NaN itu terjadi gara2 misal contoh terjadi transfer internal ya automatis counterparty_name dan kolom2 bersangkutan akan jadi NaN di SQL. Begitu juga sebaliknya yang terjadi jika yang terjadi transfer external. 

</br>

Akhirnya saya pakai COALESCE karena untuk Null handling dan untuk punya kolom penerima yang lebih seragam contoh: 
- receiver_party_id
- receiver_party_name
- receiver_party_address
- receiver_party_country
- receiver_party_risk_level

COALESCE mengambil nilai pertama yang tidak kosong, dari kiri ke kanan.
Contoh:
coalesce(
    receiver_customer.full_name,
    counterparty.counterparty_name,
    t.receiver_name
) as receiver_party_name
Logikanya:
1. Jika penerima adalah customer internal, gunakan nama dari customers.
2. Jika bukan internal tetapi external counterparty tersedia, gunakan counterparties.counterparty_name.
3. Jika master data tidak tersedia, gunakan nama yang tercatat pada event transaksi: t.receiver_name.

Contoh nyata <b>COALESCE</b></br>
<b>Transfer internal</b></br>
receiver_customer.full_name     = "Anisa Santoso"
counterparty.counterparty_name  = NULL
t.receiver_name                 = "Anisa Santoso"
</br>
Hasil:
receiver_party_name = "Anisa Santoso"
</br>
<b>Transfer eksternal</b>
</br>
receiver_customer.full_name     = NULL
counterparty.counterparty_name  = "Rani Prakoso"
t.receiver_name                 = "Rani Prakoso"
Hasil: </br>
receiver_party_name = "Rani Prakoso"
Jadi kedua tipe transaksi sekarang punya satu kolom penerima yang konsisten.
</br>


In [6]:
joined_tables.columns

Index(['transaction_id', 'transaction_timestamp', 'transaction_type',
       'channel', 'transaction_status', 'debit_credit', 'amount', 'currency',
       'amount_idr_equivalent', 'purpose_code', 'purpose_description',
       'reference_number', 'source_of_fund', 'destination_bank',
       'destination_country', 'device_id', 'ip_address', 'latitude',
       'longitude', 'sender_customer_id', 'sender_account_id',
       'sender_customer_name', 'sender_customer_address',
       'sender_customer_country', 'sender_customer_monthly_income',
       'sender_customer_occupation', 'sender_customer_segment',
       'sender_customer_risk_rating', 'sender_customer_pep_flag',
       'sender_customer_onboarding_date', 'sender_account_type',
       'sender_account_currency', 'sender_branch_code',
       'sender_account_opening_date', 'sender_account_status',
       'sender_account_risk_level', 'receiver_party_id', 'receiver_party_name',
       'receiver_party_address', 'receiver_party_country',
     

In [27]:
view = ["sender_customer_monthly_income", "sender_customer_id", "sender_customer_risk_rating", "sender_customer_pep_flag",  "receiver_customer_id", "receiver_account_id", "receiver_party_name", "receiver_party_id"]
joined_tables[view].head(10)

,sender_customer_monthly_income,sender_customer_id,sender_customer_risk_rating,sender_customer_pep_flag,receiver_customer_id,receiver_account_id,receiver_party_name,receiver_party_id
0,5001000.0,CUS0004481,Low,0,CUS0007767,ACC00003390,Rizky Chandra,CUS0007767
1,6368000.0,CUS0007453,High,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Gita Adinata,CP0004049
2,18653000.0,CUS0005894,Medium,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Raka Santoso,CP0003095
3,10614000.0,CUS0000187,Low,0,CUS0000366,ACC00008341,Indra Adinata,CUS0000366
4,14113000.0,CUS0004923,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Citra Santoso,CP0001839
5,5110000.0,CUS0009012,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Bima Kurniawan,CP0001201
6,9494000.0,CUS0007860,Medium,0,CUS0003329,ACC00007330,Farah Wibowo,CUS0003329
7,27469000.0,CUS0005278,Low,0,CUS0005801,ACC00012038,Anisa Iskandar,CUS0005801
8,16129000.0,CUS0008434,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,Eka Hartono,CP0003878
9,22418000.0,CUS0008497,Low,0,EXTERNAL_NOT_BANK_CUSTOMER,EXTERNAL_ACCOUNT_NOT_ON_US,PT Darmawan Synthetic,CP0000835


In [8]:
joined_tables.to_csv(
    PROJECT_ROOT / "notebooks" / "joined_tables.csv",
    index=False,
    encoding="utf-8",
)